# ENSPRESO biomethane exploration

**Goal:** derive defensible per-country biomethane potentials and costs for integration into the POMMES energy-system LP. Builds on `clever.fetch.fetch_enspreso_biomass()` + `clever.biomethane.load_enspreso_workbook()`.

**Source:**
- JRC ENSPRESO BIOMASS dataset (CC BY 4.0), DOI: 10.2905/JRC.44AZBC8
- Direct URL: https://cidportal.jrc.ec.europa.eu/ftp/jrc-opendata/ENSPRESO/ENSPRESO_BIOMASS.xlsx
- File `Last-Modified`: 2021-06-06 (catalogue last refreshed 2023-11-03)
- Reference paper: Ruiz et al. 2019, [JRC Repository](https://publications.jrc.ec.europa.eu/repository/handle/JRC116900)

**Sections:**
1. Fetch + load
2. Sheet inventory + structure
3. Feedstock taxonomy (Tier 1 vs Tier 2)
4. ENSPRESO scenario definitions
5. Per-country potentials — bar charts
6. Cost analysis — supply curves
7. EU envelope — 6-cell summary + CCGT-equivalent GW
8. H₂ shadow price benchmark (from existing diagnostics)
9. **Headline: LCOE comparison panel** (Nuclear vs Electrolyser-H₂ vs ATR-Bio-H₂ vs Direct-Bio-CCGT)
10. Output CSV for model.py consumption
11. Sustainability / caveats / next steps


## 1. Fetch + load

`fetch_enspreso_biomass()` is idempotent + atomic — safe to call repeatedly. First call downloads ~15.7 MB. Subsequent calls return the cached path. `load_enspreso_workbook()` is `@lru_cache`-decorated.


In [ ]:
from __future__ import annotations
from pathlib import Path
import sys, os, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")  # headless on inari
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# Workspace detection (works on inari + local Mac)
_INARI = Path("/diskdata/cired/brigode/clever-work").exists()
WS = Path("/diskdata/cired/brigode/clever-work") if _INARI else Path("~/Desktop/clever-work").expanduser()
sys.path.insert(0, str(WS))
print(f"Workspace: {WS}  (inari={_INARI})")

from clever.fetch import fetch_enspreso_biomass
from clever.biomethane import (
    load_enspreso_workbook,
    get_country_potentials,
    summarize_eu_envelope,
    implied_ccgt_GW,
    BIOLOW_CODES, BIOMED_CODES, SCOPE_CODES,
    DEFAULT_ENSPRESO_SCENARIO,
    MODEL_AREAS,
)

# Output dirs for this notebook
TABLES_OUT = WS / "tables" / "enspreso"
FIGS_OUT   = WS / "figures" / "enspreso"
TABLES_OUT.mkdir(parents=True, exist_ok=True)
FIGS_OUT.mkdir(parents=True, exist_ok=True)

# Fetch (cached) + load
path = fetch_enspreso_biomass()
wb = load_enspreso_workbook(path)
print(f"Workbook loaded: {path}  ({path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"Sheets in memory: {list(wb.keys())}")


## 2. Sheet inventory + structure

The two sheets that matter for our analysis:
- `ENER - NUTS0 EnergyCom` (15,303 rows): per (year × scenario × country × feedstock) potential in PJ
- `COST - NUTS0 EnergyCom` (11,232 rows): per (year × scenario × country × feedstock) cost in Euro2010/GJ

Already filtered (in `load_enspreso_workbook`) to 30 model areas and with country codes mapped to model convention (EL→GR, UK→GB).


In [ ]:
print(f"=== Potential data ({wb['potential'].shape[0]:,} rows after filtering) ===")
print(f"  columns:    {list(wb['potential'].columns)}")
print(f"  years:      {sorted(wb['potential']['Year'].unique())}")
print(f"  scenarios:  {sorted(wb['potential']['Scenario'].unique())}")
print(f"  feedstocks: {sorted(wb['potential']['Energy Commodity'].unique())}")
print(f"  countries:  {sorted(wb['potential']['country'].unique())}")
print(f"  units:      {wb['potential']['units'].unique()}")
print()
print(f"=== Cost data ({wb['cost'].shape[0]:,} rows after filtering) ===")
print(f"  columns:    {list(wb['cost'].columns)}")
print(f"  scenarios:  {sorted(wb['cost']['Scenario'].unique())}")
print(f"  units:      {wb['cost']['Units'].unique()}")


## 3. Feedstock taxonomy

ENSPRESO breaks biomass into 17 commodities. For "biomethane" specifically (vs woody combustion, vs bioliquids), we use this two-tier classification:

### Tier 1 — Clearly biomethane (waste-derived, uncontested)

| Code | Description |
|---|---|
| `MINBIOGAS1` | **Manure (solid + liquid)** — primary AD substrate |
| `MINBIOAGRW1` | Agricultural waste |
| `MINBIOFRSR1a` | Landscape-care residues |
| `MINBIOMUN1` | **Municipal waste** — biogas via AD + landfill |
| `MINBIOSLU1` | Sewage sludge — wastewater treatment AD |

### Tier 2 — Add lignocellulosic AD-substrate (contested edge)

| Code | Description |
|---|---|
| `MINBIOCRP31` | Miscanthus, switchgrass, RCG — can be AD substrate but typically combusted |

### Excluded (not biomethane pathway)

Energy crops for bioethanol/biodiesel (`MINBIOCRP11/21/41`, `MINBIOLIQ1`, `MINBIORPS1`) and forestry combustion residues (`MINBIOFRSR1`, `MINBIOWOO*`) — different conversion technologies, different products.

The CLEVER worldview uses Tier 1 only (sufficiency-aligned: only mobilise waste streams). The policy worldview uses Tier 1+2 (REPowerEU-aligned: also accept dedicated lignocellulosic crops on marginal land).


In [ ]:
# Show the glossary for transparency
gloss = wb["glossary"][["commodity", "description", "group"]].dropna()
gloss["bio_scope"] = gloss["commodity"].apply(
    lambda c: "Tier 1 (bioLow ∪ bioMed)" if c in BIOLOW_CODES
              else ("Tier 2 (bioMed only)" if c in (BIOMED_CODES - BIOLOW_CODES)
                    else "excluded")
)
print(gloss.to_string(index=False))


## 4. ENSPRESO scenario definitions

ENSPRESO offers three availability scenarios per the JRC's biomass mobilisation assumptions:

| ENSPRESO scenario | Mobilisation philosophy |
|---|---|
| `ENS_Low` | Conservative — only readily mobilisable resources |
| `ENS_Med` | Mainstream — standard policy-aligned mobilisation |
| `ENS_High` | Aggressive — full technical potential including contested categories |

For the paired-scenario design:
- CLEVER worldview (sufficiency, low demand) ↔ `ENS_Low` × `bioLow` scope
- Policy worldview (REPowerEU, demand uplift) ↔ `ENS_Med` × `bioMed` scope

Each ENSPRESO scenario also has a "Forest" sub-variant (`*_ForestBaU`, `*_Forest400Mm3`) that only differs in woody-feedstock assumptions — irrelevant for biomethane.


## 5. Per-country biomethane potentials

Side-by-side: paired CLEVER (ENS_Low × bioLow) vs paired Policy (ENS_Med × bioMed). Sorted by descending paired-policy potential.


In [ ]:
df_clever = get_country_potentials(scope="bioLow", enspreso_scenario="ENS_Low",
                                    year=2050, wb=wb)
df_policy = get_country_potentials(scope="bioMed", enspreso_scenario="ENS_Med",
                                    year=2050, wb=wb)

# Merge for side-by-side
merged = df_policy[["country", "potential_TWh_th"]].rename(
    columns={"potential_TWh_th": "policy_bioMed"}
).merge(
    df_clever[["country", "potential_TWh_th"]].rename(
        columns={"potential_TWh_th": "CLEVER_bioLow"}),
    on="country", how="outer"
).fillna(0)
merged = merged.sort_values("policy_bioMed", ascending=False).reset_index(drop=True)

# Bar chart
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(merged))
w = 0.4
ax.bar(x - w/2, merged["policy_bioMed"],  width=w, label="Policy worldview\n(bioMed × ENS_Med)", color="firebrick")
ax.bar(x + w/2, merged["CLEVER_bioLow"], width=w, label="CLEVER worldview\n(bioLow × ENS_Low)", color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(merged["country"], rotation=60, ha="right")
ax.set_ylabel("Biomethane potential, 2050 (TWh_th/yr)")
ax.set_title("Per-country biomethane potential — paired worldviews")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
out = FIGS_OUT / "per_country_potential_paired"
fig.savefig(f"{out}.png", dpi=120); fig.savefig(f"{out}.svg"); plt.close()
print(f"→ wrote {out}.{{png,svg}}")
print()
print("Top 10 by paired-policy potential:")
print(merged.head(10).to_string(index=False))


## 6. Cost analysis — supply curves

ENSPRESO costs are **feedstock gate prices** in Euro2010/GJ. We convert to Euro2010/MWh_th (× 3.6) and inflate to ~2024 (× 1.4 CPI). Note these costs do NOT include AD plant capex — see Section 9 for the full delivered-biomethane LCOE.


In [ ]:
# Per-country weighted cost (already computed) + tier breakdown
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: cost vs potential scatter (CLEVER pairing)
axes[0].scatter(df_clever["potential_TWh_th"], df_clever["cost_eur_per_MWh_th"],
                s=80, c="steelblue", alpha=0.7, edgecolor="black")
for _, r in df_clever.iterrows():
    if r["potential_TWh_th"] > 5:
        axes[0].annotate(r["country"], (r["potential_TWh_th"], r["cost_eur_per_MWh_th"]),
                         fontsize=8, alpha=0.8)
axes[0].set_xlabel("Per-country potential (TWh_th/yr)")
axes[0].set_ylabel("Weighted feedstock cost (€2024/MWh_th)")
axes[0].set_title("CLEVER pairing: bioLow × ENS_Low")
axes[0].grid(alpha=0.3)

# Right: same for policy
axes[1].scatter(df_policy["potential_TWh_th"], df_policy["cost_eur_per_MWh_th"],
                s=80, c="firebrick", alpha=0.7, edgecolor="black")
for _, r in df_policy.iterrows():
    if r["potential_TWh_th"] > 10:
        axes[1].annotate(r["country"], (r["potential_TWh_th"], r["cost_eur_per_MWh_th"]),
                         fontsize=8, alpha=0.8)
axes[1].set_xlabel("Per-country potential (TWh_th/yr)")
axes[1].set_ylabel("Weighted feedstock cost (€2024/MWh_th)")
axes[1].set_title("Policy pairing: bioMed × ENS_Med")
axes[1].grid(alpha=0.3)

plt.tight_layout()
out = FIGS_OUT / "cost_vs_potential_scatter"
fig.savefig(f"{out}.png", dpi=120); fig.savefig(f"{out}.svg"); plt.close()
print(f"→ wrote {out}.{{png,svg}}")
print()
print(f"Per-country cost statistics (paired CLEVER bioLow × ENS_Low):")
print(f"  mean   = {df_clever['cost_eur_per_MWh_th'].mean():.1f} €/MWh_th")
print(f"  median = {df_clever['cost_eur_per_MWh_th'].median():.1f}")
print(f"  range  = [{df_clever['cost_eur_per_MWh_th'].min():.1f}, {df_clever['cost_eur_per_MWh_th'].max():.1f}]")
print()
print(f"Per-country cost statistics (paired Policy bioMed × ENS_Med):")
print(f"  mean   = {df_policy['cost_eur_per_MWh_th'].mean():.1f} €/MWh_th")
print(f"  median = {df_policy['cost_eur_per_MWh_th'].median():.1f}")
print(f"  range  = [{df_policy['cost_eur_per_MWh_th'].min():.1f}, {df_policy['cost_eur_per_MWh_th'].max():.1f}]")


## 7. EU envelope — 6-cell summary

What the LP would see, EU-wide. Includes implied "if all biomethane is dispatched at 80% CF" CCGT-equivalent nameplate.


In [ ]:
env = summarize_eu_envelope(year=2050, wb=wb)
env["GW_CCGT_at_80CF"] = env["EU_potential_TWh_th"].apply(implied_ccgt_GW)
env_disp = env[["enspreso_scenario", "scope", "EU_potential_TWh_th",
                "GW_CCGT_at_80CF", "EU_weighted_cost_eur_per_MWh_th",
                "n_countries_with_supply"]].copy()
env_disp.columns = ["ENSPRESO", "scope", "TWh_th/yr", "GW CCGT @ 80% CF",
                    "€/MWh_th (weighted)", "n countries"]
env_disp.to_csv(TABLES_OUT / "eu_envelope_2050.csv", index=False)
print(env_disp.round(1).to_string(index=False))
print()
print("Paired choices for article:")
print(f"  CLEVER:  ENS_Low × bioLow → {env.loc[(env.enspreso_scenario=='ENS_Low')&(env.scope=='bioLow'), 'EU_potential_TWh_th'].iloc[0]:.0f} TWh_th, {env.loc[(env.enspreso_scenario=='ENS_Low')&(env.scope=='bioLow'), 'GW_CCGT_at_80CF'].iloc[0]:.0f} GW CCGT-equivalent")
print(f"  Policy:  ENS_Med × bioMed → {env.loc[(env.enspreso_scenario=='ENS_Med')&(env.scope=='bioMed'), 'EU_potential_TWh_th'].iloc[0]:.0f} TWh_th, {env.loc[(env.enspreso_scenario=='ENS_Med')&(env.scope=='bioMed'), 'GW_CCGT_at_80CF'].iloc[0]:.0f} GW CCGT-equivalent")


## 8. H₂ shadow price benchmark (from existing diagnostics)

Before adding biomethane to the model, what does the LP currently say H₂ costs in the existing solutions? This anchors the LCOE comparison in Section 9 — we know what biomethane has to BEAT to be picked over electrolyser-H₂.


In [ ]:
# Pull from policy_nuke dual file if available
dual_path = WS / "results" / "diagnostics" / "policy_nuke" / "dual_2050.nc"
if not dual_path.exists():
    # local Mac fallback
    dual_path = WS / "R0_v1" / "diagnostics" / "dual_2050.nc"

if dual_path.exists():
    dual = xr.open_dataset(dual_path)
    ac = dual["operation_adequacy_constraint"]
    h2 = np.abs(ac.sel(resource="hydrogen").values.flatten())
    h2 = h2[~np.isnan(h2)]
    elec = np.abs(ac.sel(resource="electricity").values.flatten())
    elec = elec[~np.isnan(elec)]

    fig, ax = plt.subplots(figsize=(11, 5))
    # Cap extremely high tails at 200 €/MWh for readability
    h2_plot = h2[h2 < 300]
    elec_plot = elec[elec < 300]
    ax.hist(h2_plot,    bins=50, alpha=0.6, label=f"H₂ ({len(h2):,} obs)",   color="seagreen", density=True)
    ax.hist(elec_plot,  bins=50, alpha=0.4, label=f"Electricity ({len(elec):,} obs)", color="steelblue", density=True)
    ax.axvline(np.median(h2),   color="seagreen",  linestyle="--", linewidth=2, label=f"H₂ median = {np.median(h2):.0f}")
    ax.axvline(np.median(elec), color="steelblue", linestyle="--", linewidth=2, label=f"Electricity median = {np.median(elec):.0f}")
    ax.set_xlabel("Shadow price (€/MWh of resource)")
    ax.set_ylabel("density")
    ax.set_title(f"Endogenous shadow prices in {dual_path.parent.name} solution")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    out = FIGS_OUT / "h2_shadow_price_distribution"
    fig.savefig(f"{out}.png", dpi=120); fig.savefig(f"{out}.svg"); plt.close()

    print(f"H₂ shadow price (€/MWh of H₂):")
    print(f"  median = {np.median(h2):.0f}   p5 = {np.percentile(h2, 5):.0f}   p95 = {np.percentile(h2, 95):.0f}")
    print(f"  → translated to electricity-from-H₂-CCGT (÷ 0.58 eff):")
    print(f"     median = {np.median(h2)/0.58:.0f}   p5 = {np.percentile(h2,5)/0.58:.0f}   p95 = {np.percentile(h2,95)/0.58:.0f}  €/MWh_e")
    dual.close()
else:
    print(f"No dual file found at {dual_path}. Skipping H₂ price benchmark.")


## 9. Headline LCOE comparison panel

The chart that defines the article's narrative on biomethane:

- Y-axis: LCOE for delivered electricity (€/MWh_e) or delivered H₂ (€/MWh_H₂)
- X-axis: assumed electrolyser CAPEX (€/kW_H₂_output)
- Lines: four competing pathways

**Pathways:**

1. **Nuclear** (horizontal — independent of electrolyser cost)
2. **Direct biomethane CCGT** (horizontal — independent)
3. **Electrolyser → H₂ → H₂-CCGT** (rises with electrolyser CAPEX)
4. **ATR-biomethane → H₂ → H₂-CCGT** (horizontal — independent)

Crossover points show where the LP's marginal H₂ supplier flips from electrolyser to ATR.


In [ ]:
# LCOE model — single source of truth for the article
# All values in €2024, capacity factors are model-design assumptions

EUR_INFL = 1.0  # values below are already in 2024-equivalent

def crf(r, n):
    return r / (1 - (1+r)**-n)

# Common parameters
DISC = 0.04
CCGT_EFF = 0.58
AD_CF = 0.90
CCGT_CF = 0.80
ELECTROLYSER_EFF = 0.75      # H₂ output / electricity input
ATR_EFF = 0.75               # H₂ output / biomethane input
ELECTRICITY_PRICE = 50.0     # €/MWh, weighted average for electrolyser input

# Tech parameters (capex €/kW of nameplate as defined, FOM €/kW/yr)
TECHS = {
    "Nuclear": dict(capex=8250, fom=103, life=60, vom=0.0083, fuel=8.0,
                    cf=0.80, kind="electric"),
    "CCGT (fossil)": dict(capex=1015, fom=47, life=30, vom=6, fuel=125.0,
                          cf=0.50, kind="electric"),
    "Electrolyser": dict(capex=500, fom=15, life=20, vom=0,
                         fuel=ELECTRICITY_PRICE/ELECTROLYSER_EFF,
                         cf=0.80, kind="h2"),
    "AD plant (Methanization)": dict(capex=2875, fom=117.875, life=25, vom=0,
                                     fuel=17.0, cf=AD_CF, kind="biomethane"),
    "ATR": dict(capex=700, fom=35, life=25, vom=5, fuel=0,  # fuel = biomethane cost added separately
                cf=0.80, kind="h2"),
    "H₂-CCGT": dict(capex=1083, fom=50, life=30, vom=7, fuel=0,  # fuel from H₂ pool
                    cf=0.50, kind="electric"),
    "Bio-CCGT (= same as fossil CCGT physically)": dict(capex=1015, fom=47, life=30, vom=6, fuel=0,
                                                        cf=0.50, kind="electric"),
}

def lcoe_simple(capex, fom, life, vom, fuel, cf, kind="electric"):
    # LCOE for a single tech in isolation, ignoring chain coupling.
    annuity = capex * crf(DISC, life)
    mwh_per_mw_yr = 8760 * cf
    return (annuity + fom) * 1000 / mwh_per_mw_yr + vom + fuel

# Standalone LCOEs (for reference table)
print("=== Standalone LCOEs (single tech, not chain) ===")
for name, p in TECHS.items():
    lcoe = lcoe_simple(**p)
    unit = "€/MWh_e" if p["kind"] == "electric" else ("€/MWh_H₂" if p["kind"] == "h2" else "€/MWh_th")
    print(f"  {name:50s}  {lcoe:6.1f}  {unit}")


In [ ]:
# Chained LCOEs — what the LP actually sees end-to-end
# We compute "delivered electricity" cost for each pathway

# Pathway 1: Direct biomethane CCGT (bundled AD + CCGT)
# Need 1.72 MWh_th biomethane per MWh_e at 58% eff
# Per MW_e CCGT @ CCGT_CF, biomethane needed/yr = (8760*CCGT_CF/CCGT_EFF) MWh
# Per MW AD @ AD_CF, biomethane delivered/yr = 8760*AD_CF MWh
# → AD nameplate per MW_e = (CCGT_CF/CCGT_EFF) / AD_CF
def lcoe_direct_bio_ccgt(ad_capex=2875, ad_fom=117.875, ad_life=25,
                         ccgt_capex=1015, ccgt_fom=47, ccgt_life=30, ccgt_vom=6,
                         feedstock_cost=17.0, ad_cf=AD_CF, ccgt_cf=CCGT_CF,
                         ccgt_eff=CCGT_EFF):
    ad_nameplate_per_MW_e = (ccgt_cf / ccgt_eff) / ad_cf
    capex_per_MW_e = ccgt_capex + ad_capex * ad_nameplate_per_MW_e
    fom_per_MW_e   = ccgt_fom   + ad_fom   * ad_nameplate_per_MW_e
    annuity = (ccgt_capex * crf(DISC, ccgt_life)
               + ad_capex * crf(DISC, ad_life) * ad_nameplate_per_MW_e)
    fuel_per_MWh_e = feedstock_cost / ccgt_eff
    mwh_per_yr = 8760 * ccgt_cf
    return (annuity + fom_per_MW_e) * 1000 / mwh_per_yr + ccgt_vom + fuel_per_MWh_e

# Pathway 2: Electrolyser → H₂-CCGT (cost varies with electrolyser CAPEX)
def lcoe_electrolyser_h2_ccgt(electrolyser_capex=500, electricity_price=50.0,
                              ccgt_capex=1083, ccgt_fom=50, ccgt_vom=7,
                              electrolyser_fom=15, electrolyser_life=20,
                              ccgt_life=30,
                              ccgt_cf=CCGT_CF, electrolyser_cf=CCGT_CF,
                              ccgt_eff=CCGT_EFF, electrolyser_eff=ELECTROLYSER_EFF):
    # H₂ needed per MWh_e: 1/ccgt_eff = 1.72
    h2_per_MW_e_yr = (8760 * ccgt_cf) / ccgt_eff  # MWh_H₂ needed per MW_e CCGT
    electrolyser_MW_per_MW_e = h2_per_MW_e_yr / (8760 * electrolyser_cf)
    electricity_input_per_MW_e_yr = h2_per_MW_e_yr / electrolyser_eff
    # Capex stack per MW_e CCGT output
    h2_ccgt_capex_per_MW_e = ccgt_capex
    electrolyser_capex_per_MW_e = electrolyser_capex * electrolyser_MW_per_MW_e
    h2_ccgt_fom_per_MW_e = ccgt_fom
    electrolyser_fom_per_MW_e = electrolyser_fom * electrolyser_MW_per_MW_e
    annuity = (h2_ccgt_capex_per_MW_e * crf(DISC, ccgt_life)
               + electrolyser_capex_per_MW_e * crf(DISC, electrolyser_life))
    fom_total = h2_ccgt_fom_per_MW_e + electrolyser_fom_per_MW_e
    # Variable cost = electricity input for H₂ production
    fuel_per_MWh_e = electricity_input_per_MW_e_yr * electricity_price / (8760 * ccgt_cf)
    mwh_per_yr = 8760 * ccgt_cf
    return (annuity + fom_total) * 1000 / mwh_per_yr + ccgt_vom + fuel_per_MWh_e

# Pathway 3: ATR-biomethane → H₂-CCGT
def lcoe_atr_bio_h2_ccgt(electrolyser_capex=None,  # not used, kept for signature
                         atr_capex=700, atr_fom=35, atr_vom=5, atr_life=25, atr_eff=ATR_EFF,
                         ad_capex=2875, ad_fom=117.875, ad_life=25,
                         ccgt_capex=1083, ccgt_fom=50, ccgt_vom=7, ccgt_life=30,
                         feedstock_cost=17.0,
                         ccgt_cf=CCGT_CF, atr_cf=CCGT_CF, ad_cf=AD_CF,
                         ccgt_eff=CCGT_EFF):
    h2_per_MW_e_yr = (8760 * ccgt_cf) / ccgt_eff
    atr_MW_per_MW_e = h2_per_MW_e_yr / (8760 * atr_cf)
    biomethane_per_MW_e_yr = h2_per_MW_e_yr / atr_eff
    ad_nameplate_per_MW_e = (biomethane_per_MW_e_yr / 8760) / ad_cf

    annuity = (ccgt_capex * crf(DISC, ccgt_life)
               + atr_capex * crf(DISC, atr_life) * atr_MW_per_MW_e
               + ad_capex * crf(DISC, ad_life) * ad_nameplate_per_MW_e)
    fom_total = (ccgt_fom
                 + atr_fom * atr_MW_per_MW_e
                 + ad_fom * ad_nameplate_per_MW_e)
    fuel_per_MWh_e = feedstock_cost * biomethane_per_MW_e_yr / (8760 * ccgt_cf)
    vom_total = ccgt_vom + atr_vom * (atr_MW_per_MW_e * ccgt_cf / atr_cf)
    mwh_per_yr = 8760 * ccgt_cf
    return (annuity + fom_total) * 1000 / mwh_per_yr + vom_total + fuel_per_MWh_e

# Pathway 4: Nuclear (standalone)
def lcoe_nuclear():
    return lcoe_simple(**TECHS["Nuclear"])

# Headline numbers
print("=== Pathway LCOEs at default assumptions (electrolyser 500 €/kW, electricity 50 €/MWh, 80% CCGT CF) ===")
print(f"  Nuclear (LCOE)                               : {lcoe_nuclear():6.1f} €/MWh_e")
print(f"  Direct Biomethane CCGT (AD + CCGT bundled)   : {lcoe_direct_bio_ccgt():6.1f} €/MWh_e")
print(f"  Electrolyser → H₂ → H₂-CCGT (chain)          : {lcoe_electrolyser_h2_ccgt():6.1f} €/MWh_e")
print(f"  ATR-biomethane → H₂ → H₂-CCGT (chain)        : {lcoe_atr_bio_h2_ccgt():6.1f} €/MWh_e")


In [ ]:
# Sensitivity: vary electrolyser CAPEX from 500 to 900 (the planned sensitivity range)
import numpy as np
ec_range = np.linspace(500, 900, 50)
lcoe_nuke   = [lcoe_nuclear() for _ in ec_range]
lcoe_bio    = [lcoe_direct_bio_ccgt() for _ in ec_range]
lcoe_elec   = [lcoe_electrolyser_h2_ccgt(electrolyser_capex=c) for c in ec_range]
lcoe_atr    = [lcoe_atr_bio_h2_ccgt() for _ in ec_range]

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(ec_range, lcoe_nuke,  label="Nuclear (capex 8,250 €/kW)",                    color="purple",    linewidth=2)
ax.plot(ec_range, lcoe_bio,   label="Direct biomethane CCGT (AD bundled)",           color="seagreen",  linewidth=2)
ax.plot(ec_range, lcoe_elec,  label="Electrolyser → H₂ → H₂-CCGT (varies)",          color="steelblue", linewidth=2)
ax.plot(ec_range, lcoe_atr,   label="ATR-biomethane → H₂ → H₂-CCGT",                 color="firebrick", linewidth=2, linestyle="--")
# Mark the three sensitivity probe points (500, 700, 900)
for pt in [500, 700, 900]:
    ax.axvline(pt, color="gray", linestyle=":", alpha=0.4)
    ax.text(pt, ax.get_ylim()[0]+5, f"el{pt}", ha="center", fontsize=8)

ax.set_xlabel("Electrolyser CAPEX (€/kW_H₂)")
ax.set_ylabel("LCOE delivered electricity (€/MWh_e)")
ax.set_title("Headline LCOE comparison @ 80% CCGT CF, 50 €/MWh electricity input, 4% discount")
ax.legend(loc="best", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out = FIGS_OUT / "lcoe_comparison_panel"
fig.savefig(f"{out}.png", dpi=120); fig.savefig(f"{out}.svg"); plt.close()
print(f"→ wrote {out}.{{png,svg}}")
print()
print("LCOE at the three sensitivity points (€/MWh_e):")
for c in [500, 700, 900]:
    print(f"  Electrolyser CAPEX = {c} €/kW:")
    print(f"    Nuclear:                {lcoe_nuclear():6.1f}")
    print(f"    Direct Bio-CCGT:        {lcoe_direct_bio_ccgt():6.1f}")
    print(f"    Electrolyser → H₂-CCGT: {lcoe_electrolyser_h2_ccgt(electrolyser_capex=c):6.1f}")
    print(f"    ATR-Bio → H₂-CCGT:      {lcoe_atr_bio_h2_ccgt():6.1f}")


## 10. Output CSV — for model.py consumption

Two CSVs, one per paired scenario, with the per-country potentials + cost the model will use:
- `tables/enspreso/potentials_bioLow_ENS_Low_2050.csv` — for `_bioLow` scenarios
- `tables/enspreso/potentials_bioMed_ENS_Med_2050.csv` — for `_bioMed` scenarios

The model's `add_biomethane_from_enspreso(country_code, scope)` function will read these.


In [ ]:
for scope, scen in [("bioLow", "ENS_Low"), ("bioMed", "ENS_Med")]:
    df = get_country_potentials(scope=scope, enspreso_scenario=scen,
                                year=2050, wb=wb, eur_year=2024)
    out_path = TABLES_OUT / f"potentials_{scope}_{scen}_2050.csv"
    df.to_csv(out_path, index=False)
    print(f"  {scope} × {scen}: {len(df)} countries, total {df['potential_TWh_th'].sum():.0f} TWh_th")
    print(f"    → {out_path}")
print()
print("CSV schema:")
print("  country               — model area code")
print("  potential_TWh_th      — annual cap in TWh thermal/yr")
print("  potential_PJ          — same, in PJ (for cross-check)")
print("  cost_eur_per_MWh_th   — volume-weighted feedstock cost in €2024/MWh_th")
print("  n_feedstocks          — how many feedstock categories contribute")
print()
print("Sample (paired Policy bioMed × ENS_Med, top 5):")
sample = pd.read_csv(TABLES_OUT / "potentials_bioMed_ENS_Med_2050.csv").head(5)
print(sample.to_string(index=False))


## 11. Sustainability + caveats + next steps

### Modelling choices and their defensibility

**Choice 1: We use only Tier 1 + Tier 2 feedstocks.** Excludes forestry combustion residues (`MINBIOWOO*`), bioethanol crops (`MINBIOCRP11/21`), and biodiesel crops (`MINBIORPS1`). Reason: these conversion pathways yield different end-products (heat, ethanol, biodiesel) — not biomethane. Including them would double-count biomass.

**Choice 2: Cost is feedstock GATE PRICE only (~17 €/MWh_th).** AD plant capex (2,875 €/kW × CRF + FOM) is added separately in the LP via the `Methanization` tech bundle (Option c). Total delivered biomethane LCOE is ~47–60 €/MWh_th (see Section 9).

**Choice 3: Per-country potentials are FIXED yearly caps with FREE intra-year flexibility.** Biomethane is treated as storable indefinitely within a year (real-world: existing CH₄ reservoirs cover this). The LP can dispatch at any hour up to the cap. Limitation: no cross-border trade — biomethane from FR cannot serve DE demand. Justified by: biomethane is mostly nationally consumed in reality; modeling cross-border trade would add complexity for second-order gains.

**Choice 4: `bioLow` paired with ENS_Low (CLEVER), `bioMed` paired with ENS_Med (policy).** Coherent narrative: low-demand-low-mobilisation vs higher-demand-mainstream-mobilisation. Off-diagonal scenarios (R0_v1_bioMed, policy_re_bioLow) included as robustness probes in the sensitivity matrix.

### Known limitations to disclose in the article

1. **ATR pathway**: optional in sensitivity set (`_atr` suffix). When disabled, biomethane can only go to direct CCGT — may bias against biomethane at high electrolyser CAPEX.
2. **Cross-border biomethane trade**: not modelled. Bias would slightly reduce biomethane utility in countries with low domestic feedstock (LU, MT, IE).
3. **Negative-emissions BECCS**: ATR+CCS not modelled. Would credit biomethane with negative emissions if reviewers raise EU 2050 net-zero feasibility.
4. **Single-year (2050)**: trajectory effects (when does biomethane mobilisation ramp?) not captured.
5. **ENSPRESO data vintage**: underlying data is 2021 (file Last-Modified); catalogue refreshed 2023. May be slightly stale vs latest national plans.

### Next steps

1. ✅ Pipeline (`fetch_enspreso_biomass` + `load_enspreso_workbook` + `get_country_potentials`)
2. ✅ Per-country potential CSVs (this notebook)
3. → Add `Biomethane_CCGT` conversion tech in `model.py` (bundled-AD pattern)
4. → Add `ATR_biomethane` tech in `model.py` (consumes biomethane, produces hydrogen resource)
5. → Suffix parser for `_bioLow`, `_bioMed`, `_atr`, `_elNNN` in `constants.py`
6. → Run 11-scenario sensitivity matrix (~22h compute, parallel-2)
7. → Update `cross_scenario_analysis.ipynb` to include biomethane comparison panels
